# LFM Semantic Segmentation Comparison Workflow

## Purpose of this notebook
This notebook runs the existing toy DINO semantic segmentation model on the full-model split directory layout used by the Graha/Lunar-FM workflow. It uses the same helper functions as the sbatch script so notebook and batch behavior stay aligned while preserving the old model's baseline behavior.

## Imports, Dino Repo Clone

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)

In [ ]:
from argparse import Namespace

from lightning.pytorch import seed_everything

from lfm.full_model.utils import create_timestamped_output_dir
from lfm.full_model.utils.utils import ensure_data_symlink
from toy_sem_seg_comparison import (
    build_config,
    create_datamodule,
    create_lightning_module,
    create_model,
    create_trainer,
    save_config,
    validate_data_paths,
)

## User Config

#### Paths
`INPUT_ROOT_DIR`: optional source directory containing the split-folder dataset. If set, the notebook creates `./data` as a symlink to this directory.

`DATA_ROOT`: optional explicit data directory. Leave as `None` to use `./data`, matching the sbatch script default.

`OUTPUT_DIR`: output root for checkpoints, config snapshots, file lists, logs, and visualizations.

`DINO_CHECKPOINT`: optional local DinoV3 checkpoint path. Leave as `None` to use the default path in `sseg_model.py`.

#### Dataset parameters
`BAND_FILTER`: list of input bands to keep, in order.

`TARGET_SIZE` and `SPATIAL_TRANSFORM`: control whether the split data is resized or cropped before entering the toy model. The comparison baseline uses a 256x256 center crop to better match the Graha/full-model path.

`MAX_*_SAMPLES`: optional per-split sample limits for smoke tests.

#### Training hyperparameters
`BATCH_SIZE`, `NUM_EPOCHS`, `BASE_LR`, and `WEIGHT_DECAY` control the training run.

#### Model hyperparameters
`FREEZE_ENCODER`: whether to keep the DinoV3 encoder frozen.

`LOSS_TYPE`: Lightning loss wrapper option for the toy comparison run.

In [ ]:
# Data paths
INPUT_ROOT_DIR = None  # Example: Path("/explore/nobackup/projects/lfm/model_inputs/my_sem_seg_split")
DATA_ROOT = None  # Leave as None to use NOTEBOOK_DIR / "data"
OUTPUT_DIR = "./outputs/toy_sem_seg_comparison"
DINO_CHECKPOINT = None  # Leave as None to use the default checkpoint in sseg_model.py

# Symlink setup
SIMLINK_DEST = INPUT_ROOT_DIR

# Dataset parameters
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
TARGET_SIZE = 256
SPATIAL_TRANSFORM = "crop"  # "crop" or "resize"
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 10
NUM_EPOCHS = 1  # Use 1 for smoke tests; set to 100 for comparison runs.
BASE_LR = 5e-5
WEIGHT_DECAY = 1e-3
LOSS_TYPE = "focal_dice"

# Model parameters
FREEZE_ENCODER = False

# Runtime controls
SEED = 42
NO_FIT = False

In [ ]:
DATA_SYMLINK = ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

In [ ]:
args = Namespace(
    data_root=DATA_ROOT,
    base_output_dir=OUTPUT_DIR,
    dino_checkpoint=DINO_CHECKPOINT,
    band_filter=BAND_FILTER,
    target_size=TARGET_SIZE,
    spatial_transform=SPATIAL_TRANSFORM,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_epochs=NUM_EPOCHS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    loss_type=LOSS_TYPE,
    freeze_encoder=FREEZE_ENCODER,
    seed=SEED,
    no_fit=NO_FIT,
)

config = build_config(args)
validate_data_paths(config)

print("Data root:", config.data_root)
print("Base output dir:", config.base_output_dir)
print("Band filter:", config.band_filter)
print("Target size:", config.target_size)
print("Spatial transform:", config.spatial_transform)
print("Max train/val/test samples:", config.max_train_samples, config.max_val_samples, config.max_test_samples)
print("Normalize inputs:", config.normalize_inputs)
print("Loss type:", config.loss_type)
print("Max epochs:", config.max_epochs)

## Create output directory

In [ ]:
output_dir = create_timestamped_output_dir(config.base_output_dir)
save_config(config, output_dir)
print("Output dir:", output_dir)

## Create dataloaders

In [ ]:
seed_everything(config.seed)
datamodule = create_datamodule(config, output_dir)

if datamodule.weight_assignments is None:
    raise RuntimeError("DataModule did not create weight assignments.")

print("Weight assignments:", datamodule.weight_assignments)

## Load Encoder and Create Model

In [ ]:
model = create_model(config, datamodule.weight_assignments)
task = create_lightning_module(config, model)

print(type(model))
print(type(task))

## Trainer

In [ ]:
trainer = create_trainer(config, output_dir)

## Run Training

In [ ]:
# Set args.max_epochs = 1 in the config cell for a smoke test.
if args.no_fit:
    print("Skipping trainer.fit() because args.no_fit is True.")
else:
    trainer.fit(task, datamodule=datamodule)

## Test Best Checkpoint

In [ ]:
if args.no_fit:
    print("Skipping trainer.test() because args.no_fit is True.")
else:
    trainer.test(task, datamodule=datamodule, ckpt_path="best")